# 1.3 精度压缩与 nnz-aware partition

## 本节目标

- 解释 BF16 storage + FP32 accumulation
- 追踪按累计 nnz 划分行区间的实现
- 在性能与正确性之间建立约束

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import platform, shutil
print("Python:", platform.python_version())
print("CMake:", shutil.which("cmake"))
print("正式路径：Ascend C FP32 RTC；FP16/BF16/persistent 仍为 Host Prototype")


## 为什么不只比较 FP32

CSR SpMV 的算术强度低，矩阵 values 和索引访存占主导。原工程把 FP32 values 转换为 FP16/BF16 位表示，乘法前恢复为 FP32，并用 FP32 累加。README 记录 BF16 相对 CPU reference 的误差约 `2e-3` 到 `8e-3`，因此它不是逐位等价优化。

行数均分不能保证 nnz 均衡。`build_nnz_aware_partitions` 根据 `row_ptr` 的累计 nnz 选择连续行边界；历史结果中的 balance ratio 接近 1。

## 观察与解释

压缩收益来自更小的 values footprint；row_ptr 和 col_idx 仍为 int32，所以完整 CSR 压缩比不是 2x。原 README 的大矩阵记录约从 84 MB 降至 64 MB，compression ratio 约 1.3125。分区只改变工作划分，不改变每行内部累加顺序。

## 课后实践

说明为什么 BF16 values 的元素宽度减半，但完整 CSR footprint 只压缩约 1.31x；同时写出报告必须保留的正确性字段。参考答案见 `answer/01.03_answer.md`。